# 10% Bavaria Parameter Sweep - Analysis

Reads `sweep_results.csv` from the most recent sweep and produces the OAT marginal-effect charts, Pareto scatter, savings histogram, enumeration-efficiency bars, pruning-mechanism breakdown, and the 2-factor heatmaps.

Spec: `docs/plans/2026-04-15-10pct-param-sweep-design.md`
Plan: `docs/plans/2026-04-15-10pct-param-sweep-plan.md`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd()
while REPO.name and not (REPO / 'matsim-libs').exists():
    REPO = REPO.parent

sweep_dirs = sorted((REPO / 'outputs').glob('sweep-10pct-*'))
if not sweep_dirs:
    raise FileNotFoundError('No sweep-10pct-* directory found in outputs/')
SWEEP = sweep_dirs[-1]
print(f'Sweep: {SWEEP}')

df = pd.read_csv(SWEEP / 'sweep_results.csv')
df = df[df.get('error', 0) == 0]
df.head()


## 1. OAT marginal effects


In [ ]:
oat = df[df['phase'] == 'oat'].copy()
baseline = oat[oat['combo_id'] == 'oat-baseline'].iloc[0]

knobs = ['search_horizon', 'max_detour_factor', 'min_drt_cost_per_km', 'inter_degree_keep_fraction']
fig, axes = plt.subplots(len(knobs), 3, figsize=(14, 3 * len(knobs)))
metrics = [('rides_total', 'Total rides'), ('wall_total_s', 'Wall time (s)'), ('max_degree_reached', 'Max degree reached')]

for i, knob in enumerate(knobs):
    sub = oat[oat[knob] != baseline[knob]].copy()
    sub = pd.concat([sub, pd.DataFrame([baseline])], ignore_index=True)
    sub = sub.sort_values(knob)
    for j, (metric, title) in enumerate(metrics):
        ax = axes[i, j]
        ax.bar(range(len(sub)), sub[metric], tick_label=[str(v) for v in sub[knob]])
        ax.set_title(f'{knob} vs {title}')
        ax.axhline(baseline[metric], color='red', linestyle='--', alpha=0.5, label='baseline')
        ax.legend()

plt.tight_layout()
plt.show()


## 2. Pareto scatter - rides vs wall time


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for phase in df['phase'].unique():
    sub = df[df['phase'] == phase]
    ax.scatter(sub['wall_total_s'], sub['rides_total'], label=phase, alpha=0.7, s=60)
    for _, row in sub.iterrows():
        ax.annotate(row['combo_id'], (row['wall_total_s'], row['rides_total']),
                    fontsize=7, alpha=0.6)
ax.set_xlabel('Wall time (s)')
ax.set_ylabel('Total rides')
ax.legend()
ax.set_title('Pareto: rides vs wall time')
plt.show()


## 3. Savings distribution by interDegreeKeepFraction

Plots p50, p90, p99 of per-ride savings vs the keep fraction, holding the problem-side knobs at baseline.


In [ ]:
sub = df[df['phase'] == 'oat'].copy()
sub = sub[sub['min_drt_cost_per_km'] == baseline['min_drt_cost_per_km']]
sub = sub[sub['search_horizon'] == baseline['search_horizon']]
sub = sub[sub['max_detour_factor'] == baseline['max_detour_factor']]
sub = sub.sort_values('inter_degree_keep_fraction')

degrees_present = sorted({int(c.split('_')[0][3:]) for c in df.columns
                          if c.startswith('deg') and c.endswith('_savings_p50')})

fig, axes = plt.subplots(1, len(degrees_present), figsize=(5 * len(degrees_present), 5), sharey=False)
if len(degrees_present) == 1:
    axes = [axes]
for ax, d in zip(axes, degrees_present):
    for stat in ('p50', 'p90', 'p99'):
        col = f'deg{d}_savings_{stat}'
        if col in sub.columns:
            ax.plot(sub['inter_degree_keep_fraction'], sub[col], marker='o', label=stat)
    ax.set_xlabel('interDegreeKeepFraction')
    ax.set_ylabel('Distance savings (m)')
    ax.set_title(f'Degree {d}')
    ax.legend()
plt.tight_layout()
plt.show()


## 4. Enumeration efficiency - orderings per ride by degree


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
deg_cols = [c for c in df.columns if c.startswith('deg') and c.endswith('_orderings_evaluated')]
degrees = sorted(int(c.split('_')[0][3:]) for c in deg_cols)
width = 0.8 / max(1, len(df))
for i, (_, row) in enumerate(df.iterrows()):
    vals = []
    for d in degrees:
        orderings = row.get(f'deg{d}_orderings_evaluated', 0)
        rides = row.get(f'deg{d}_rides', 1) or 1
        vals.append(orderings / rides)
    xs = [d + i * width for d in degrees]
    ax.bar(xs, vals, width=width, label=row['combo_id'])
ax.set_xlabel('Degree')
ax.set_ylabel('Orderings evaluated per ride built')
ax.set_title('Enumeration efficiency per combo per degree')
ax.legend(fontsize=6, ncol=3)
plt.show()


## 5. Pruning mechanism breakdown


In [ ]:
mechanisms = ['pruned_by_travel_time', 'pruned_by_dropoff',
              'pruned_by_delay_window_origin', 'pruned_by_delay_window_dropoff',
              'bnb_origin_skipped', 'bnb_dest_skipped']
for degree in degrees:
    fig, ax = plt.subplots(figsize=(10, 4))
    bottom = None
    xs = range(len(df))
    for mech in mechanisms:
        col = f'deg{degree}_{mech}'
        if col not in df.columns:
            continue
        vals = df[col].fillna(0).values
        ax.bar(xs, vals, bottom=bottom, label=mech)
        bottom = vals if bottom is None else bottom + vals
    ax.set_xticks(list(xs))
    ax.set_xticklabels(df['combo_id'], rotation=45, ha='right', fontsize=7)
    ax.set_title(f'Degree {degree}: pruning mechanism breakdown')
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


## 6. 2-factor heatmaps (searchHorizon x maxDetourFactor)


In [ ]:
cross = df[df['phase'].isin(['oat', 'cross'])].copy()
cross = cross[cross['min_drt_cost_per_km'] == baseline['min_drt_cost_per_km']]
cross = cross[cross['inter_degree_keep_fraction'] == baseline['inter_degree_keep_fraction']]

metrics = [('rides_total', 'Rides'), ('wall_total_s', 'Wall time (s)'), ('max_degree_reached', 'Max degree')]
fig, axes = plt.subplots(1, len(metrics), figsize=(15, 4))

for ax, (metric, title) in zip(axes, metrics):
    pivot = cross.pivot_table(index='max_detour_factor', columns='search_horizon', values=metric, aggfunc='mean')
    im = ax.imshow(pivot.values, aspect='auto', origin='lower')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('searchHorizon')
    ax.set_ylabel('maxDetourFactor')
    ax.set_title(title)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            if pd.notna(v):
                label = f'{v:.0f}' if metric != 'wall_total_s' else f'{v:.1f}'
                ax.text(j, i, label, ha='center', va='center', color='white', fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
